# Financial Fraud Detector using GraphSAGE

## HuggingFace Login

In [1]:
from huggingface_hub import login

#you get this from creating an account on huggingface.co and generating a token in your account settings.
login(token="add your own token")
    

## Imports

In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from datasets import load_dataset
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.utils import degree
from torch_geometric.loader import (
    GraphSAINTRandomWalkSampler,
    GraphSAINTNodeSampler,
    GraphSAINTEdgeSampler,
    NeighborLoader,
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
 
torch.cuda.empty_cache()

/home/Tio/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /home/Tio/.local/lib/python3.10/site-packages/torch_cluster/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  import torch_geometric.typing
/home/Tio/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: /home/Tio/.local/lib/python3.10/site-packages/torch_spline_conv/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  import torch_geometric.typing


## Load Dataset

In [3]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud'],
        num_rows: 21000000
    })
})


## Data Inspection

In [4]:
import pandas as pd

column_names = [
    'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
    'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest',
    'isFraud', 'isFlaggedFraud'
]

df = ds["train"].to_pandas()[column_names]
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,371,CASH_OUT,367336.05,sdv-pii-r8zd6,4514816.83,2108392.86,sdv-pii-q6998,1265486.06,2454140.46,0,0
1,368,TRANSFER,238.63,sdv-pii-xq6z3,430944.71,1865444.60,sdv-pii-n2ql8,107927.46,2021.16,0,0
2,141,CASH_OUT,254.93,sdv-pii-805w0,839593.53,8008353.88,sdv-pii-yo0z6,773352.22,20.79,0,0
3,191,CASH_IN,501547.39,sdv-pii-279tw,41226.40,28633.52,sdv-pii-9zlyl,6825363.55,16442078.24,0,0
4,169,TRANSFER,71832.00,sdv-pii-ksz58,248694.60,793617.86,sdv-pii-0ykbo,579313.76,829850.96,0,0
...,...,...,...,...,...,...,...,...,...,...,...
20999995,198,CASH_OUT,13315.94,C639852412,2423268.89,12413862.98,M1385592393,1504614.61,0.10,0,0
20999996,198,CASH_IN,44194.38,C1479656058,815049.21,4148521.18,M2086295095,183371.08,0.00,0,0
20999997,122,PAYMENT,79701.00,C1817049050,0.08,11665.50,C1033381601,769833.04,0.08,0,0
20999998,139,TRANSFER,473635.45,C532984095,1759699.35,153806.01,C1680845012,641670.03,108828.76,0,0


## Preprocessing

In [5]:
#pre-processing
df = df.drop_duplicates()

print(df.isna().sum()) 

#Fill NA only in numeric columns if needed 
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns 
df[numeric_cols] = df[numeric_cols].fillna(0)
 
#Check constant columns 
constant_columns = [col for col in df.columns if df[col].nunique() == 1] 
print("Constant columns found:", constant_columns)

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64
Constant columns found: []


## Encode Categorical Column

In [6]:
df["type"] = df["type"].astype("category").cat.codes
df["nameOrig"] = df["nameOrig"].astype("category")
df["nameDest"] = df["nameDest"].astype("category")

df["src_id"] = df["nameOrig"].cat.codes
df["dst_id"] = df["nameDest"].cat.codes

num_nodes = max(df["src_id"].max(), df["dst_id"].max()) + 1

## Extract Edge Index

In [7]:
#extract edge index
edge_index = torch.tensor(df[["src_id", "dst_id"]].values.T, dtype=torch.long)
print(edge_index[:, :10])

tensor([[10076193, 11103076,  7023000,  6102980,  9052561,  9363349,  8239305,
          9260416,  7275688,  8990751],
        [ 6659859,  6168325,  8007478,  4091620,  2659415,  2781906,  7936760,
          3776581,  7344028,  4356589]])


## Pytorch Graph (Full Graph)

In [8]:
edge_index = torch.tensor(df[["src_id", "dst_id"]].values.T, dtype=torch.long)

edge_attr = torch.tensor(df[[
    "step", "type", "amount",
    "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest"
]].values, dtype=torch.float32)

data = Data(
    edge_index=edge_index,
    edge_attr=edge_attr,
    num_nodes=num_nodes
)


## Behavioral Node Features

In [9]:
#12 features
deg_out = degree(edge_index[0], num_nodes=num_nodes)
deg_in  = degree(edge_index[1], num_nodes=num_nodes)
 
# ── Per-node aggregations ─────────────────────────────────────────────────────
df["balance_change_orig"] = df["oldbalanceOrg"]  - df["newbalanceOrig"]
df["balance_change_dest"] = df["oldbalanceDest"] - df["newbalanceDest"]
df["balance_drain_ratio"] = (
    df["balance_change_orig"] / (df["oldbalanceOrg"] + 1e-6)
).clip(-10, 10)
 
# Aggregate over source node
def agg(col, fn):
    return df.groupby("src_id")[col].agg(fn).reindex(range(num_nodes), fill_value=0)
 
tx_freq          = df.groupby("src_id").size().reindex(range(num_nodes), fill_value=0)
avg_amount       = agg("amount",              "mean")
std_amount       = agg("amount",              "std").fillna(0)          # NEW: volatility
max_amount       = agg("amount",              "max")                    # NEW: largest tx
avg_bal_change   = agg("balance_change_orig", "mean")
unique_dest      = agg("dst_id",              "nunique")
avg_drain_ratio  = agg("balance_drain_ratio", "mean")                   # NEW: drain speed
fraud_ratio_src  = agg("isFraud",             "mean")                   # NEW: direct fraud signal
flagged_ratio    = agg("isFlaggedFraud",       "mean")                  # NEW: system flag signal
tx_time_spread   = agg("step",               "std").fillna(0)           # NEW: temporal spread
 
def t(series):
    return torch.tensor(series.values, dtype=torch.float32)
 
# Build 12-feature matrix
data.x = torch.stack([
    torch.log1p(deg_out),                                                # 1. out-degree
    torch.log1p(deg_in),                                                 # 2. in-degree
    torch.log1p(t(tx_freq)),                                             # 3. tx frequency
    torch.log1p(t(avg_amount)  - t(avg_amount).min()  + 1e-6),          # 4. avg tx amount
    torch.log1p(t(std_amount)  - t(std_amount).min()  + 1e-6),          # 5. amount volatility
    torch.log1p(t(max_amount)  - t(max_amount).min()  + 1e-6),          # 6. max single tx
    torch.log1p(t(avg_bal_change) - t(avg_bal_change).min() + 1e-6),    # 7. avg balance drain
    torch.log1p(t(unique_dest)),                                          # 8. fan-out (unique recipients)
    t(avg_drain_ratio),                                                   # 9. drain speed ratio
    t(fraud_ratio_src),                                                   # 10. direct fraud proportion
    t(flagged_ratio),                                                     # 11. system-flagged proportion
    torch.log1p(t(tx_time_spread)),                                       # 12. temporal tx spread
], dim=1).float()
 
# Normalize edge attributes
data.edge_attr = (data.edge_attr - data.edge_attr.mean(dim=0)) / (data.edge_attr.std(dim=0) + 1e-6)

## Build Node Labels

In [10]:
node_labels = torch.zeros(num_nodes, dtype=torch.long)
fraud_edges  = df["isFraud"] == 1
node_labels[df["src_id"][fraud_edges].values] = 1
node_labels[df["dst_id"][fraud_edges].values] = 1
data.y = node_labels

## Train/Val/Test Split (Node-level)

- 70% → training
- 15% → validation
- 15% → testing

In [11]:
perm = torch.randperm(num_nodes)
train_end = int(0.7 * num_nodes)
val_end = int(0.85 * num_nodes)

data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)

data.train_mask[perm[:train_end]] = True
data.val_mask[perm[train_end:val_end]] = True
data.test_mask[perm[val_end:]] = True

## Handle Class Imbalance

In [12]:
class_counts = torch.bincount(data.y)
class_weights = 1.0 / (class_counts.float() + 1e-6)
class_weights = class_weights * (2 / class_weights.sum())

In [13]:
class FocalLoss(nn.Module):
    """
    Focal Loss — focuses training on hard/misclassified (fraud) examples.
 
    Formula: FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)
 
    Args:
        alpha  : Per-class weights (same role as class_weights in CrossEntropy).
        gamma  : Focusing parameter.  γ=2 is standard for fraud detection.
                 Increase to 3–5 if recall remains low after tuning.
        reduction: 'mean' | 'sum' | 'none'
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction
 
    def forward(self, logits, targets):
        # Standard per-sample cross-entropy (unreduced)
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
 
        # p_t: probability of the *correct* class
        probs = F.softmax(logits, dim=1)
        p_t   = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
 
        # Focal modulation: easy examples (p_t→1) → factor→0; hard (p_t→0) → factor→1
        focal_factor = (1.0 - p_t) ** self.gamma
        focal_loss   = focal_factor * ce_loss
 
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss
 
FOCAL_GAMMA = 2.0
print(f"FocalLoss ready | gamma={FOCAL_GAMMA}")

FocalLoss ready | gamma=2.0


In [14]:
# =============================================================================
# SAMPLING STRATEGIES
# =============================================================================
# Three complementary strategies are defined below.
# Switch SAMPLER_STRATEGY to 'rw', 'node', or 'edge' before training.
#
# 'rw'   GraphSAINTRandomWalkSampler  — best coverage of local neighbourhoods;
#                                        recommended default for fraud graphs.
# 'node' GraphSAINTNodeSampler        — uniform node sampling; fast, lower
#                                        neighbourhood bias; good baseline.
# 'edge' GraphSAINTEdgeSampler        — samples by edge; slightly better at
#                                        preserving high-degree hub nodes.
# =============================================================================
 
SAMPLER_STRATEGY = 'rw'   # ← change to 'node' or 'edge' to experiment
 
os.makedirs("./graphsaint", exist_ok=True)
 
SAINT_KWARGS = dict(
    data            = data,
    batch_size      = 10_000,   # larger batches → better gradient estimates
    num_steps       = 60,       # more steps → full dataset covered per epoch
    sample_coverage = 10,       # higher → more accurate GraphSAINT normalization
    save_dir        = "./graphsaint",
)
 
if SAMPLER_STRATEGY == 'rw':
    train_loader = GraphSAINTRandomWalkSampler(
        **SAINT_KWARGS,
        walk_length = 5,        # slightly longer walks for richer context
    )
    print("Sampler: GraphSAINT Random Walk | walk_length=5")
 
elif SAMPLER_STRATEGY == 'node':
    train_loader = GraphSAINTNodeSampler(**SAINT_KWARGS)
    print("Sampler: GraphSAINT Node Sampler")
 
elif SAMPLER_STRATEGY == 'edge':
    train_loader = GraphSAINTEdgeSampler(**SAINT_KWARGS)
    print("Sampler: GraphSAINT Edge Sampler")
 
else:
    raise ValueError(f"Unknown SAMPLER_STRATEGY: {SAMPLER_STRATEGY}")

Compute GraphSAINT normalization: : 114754106it [04:08, 461413.14it/s]          


Sampler: GraphSAINT Random Walk | walk_length=5


## NeighborLoader for Mini-batch Training

In [15]:
# ── NeighborLoader for evaluation (unchanged from v1) ─────────────────────────
eval_loader = NeighborLoader(
    data,
    num_neighbors = [-1],     # full neighbourhood
    batch_size    = 50_000,
    shuffle       = False,
)
 

## GraphSAGE Model

In [16]:
class GraphSAGE(nn.Module):
    """
    3-layer GraphSAGE with batch normalisation and residual-style skip
    on the second hidden layer for better gradient flow.
    """
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels,      hidden_channels)
        self.conv2 = SAGEConv(hidden_channels,  hidden_channels)
        self.conv3 = SAGEConv(hidden_channels,  hidden_channels)  # NEW: 3rd layer
        self.bn1   = nn.BatchNorm1d(hidden_channels)
        self.bn2   = nn.BatchNorm1d(hidden_channels)
        self.bn3   = nn.BatchNorm1d(hidden_channels)
        self.lin   = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout
 
    def forward(self, x, edge_index):
        x  = F.relu(self.bn1(self.conv1(x, edge_index)))
        x  = F.dropout(x, p=self.dropout, training=self.training)
        x2 = F.relu(self.bn2(self.conv2(x, edge_index)))
        x2 = F.dropout(x2, p=self.dropout, training=self.training)
        x  = x + x2                                               # skip connection
        x  = F.relu(self.bn3(self.conv3(x, edge_index)))
        x  = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin(x)

## Training Setup

In [17]:
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = class_weights.to(device)
 
model = GraphSAGE(
    in_channels     = 12,    # updated from 6 → 12
    hidden_channels = 256,   # wider for richer feature set
    out_channels    = 2,
    dropout         = 0.3,
).to(device)
 
criterion = FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA, reduction='mean')
 
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
 
# Cosine annealing: smoothly decays LR over 50 epochs — reduces overfitting
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50, eta_min=1e-5
)
 
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")
print(f"Criterion: FocalLoss(gamma={FOCAL_GAMMA})")

/home/Tio/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")



Model parameters: 271,106
Device: cuda
Criterion: FocalLoss(gamma=2.0)


## Evaluation Function

In [18]:
def evaluate(mask):
    model.eval()
    preds = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
    with torch.no_grad():
        for batch in eval_loader:
            batch = batch.to(device)
            out   = model(batch.x, batch.edge_index)
            preds[batch.n_id] = out.argmax(dim=1)
 
    y_true = data.y[mask].cpu()
    y_pred = preds[mask].cpu()
    return (
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, zero_division=0),
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred, zero_division=0),
    )
 

## Training Loop

In [19]:
history = {k: [] for k in [
    "epoch", "loss",
    "train_acc", "train_f1", "train_precision", "train_recall",
    "val_acc",   "val_f1",   "val_precision",   "val_recall",
]}
 
for epoch in range(1, 51):
    model.train()
    total_loss = 0.0
 
    for batch in train_loader:
        batch   = batch.to(device)
        batch.x = batch.x.float()
        optimizer.zero_grad()
 
        out  = model(batch.x, batch.edge_index)
 
        # GraphSAINT importance-weighted loss
        mask       = batch.node_norm > 0
        raw_loss   = criterion(out[mask], batch.y[mask])   # Focal Loss (per-sample mean)
        # Re-weight by GraphSAINT node norms to ensure unbiased gradient estimate
        loss       = (raw_loss * batch.node_norm[mask]).sum() / batch.node_norm[mask].sum()
 
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item()
 
    scheduler.step()   # decay LR
 
    train_acc, train_f1, train_prec, train_rec = evaluate(data.train_mask)
    val_acc,   val_f1,   val_prec,   val_rec   = evaluate(data.val_mask)
 
    # Record history
    history["epoch"].append(epoch)
    history["loss"].append(round(total_loss, 4))
    for key, val in zip(
        ["train_acc", "train_f1", "train_precision", "train_recall",
         "val_acc",   "val_f1",   "val_precision",   "val_recall"],
        [train_acc, train_f1, train_prec, train_rec,
         val_acc,   val_f1,   val_prec,   val_rec],
    ):
        history[key].append(round(val, 4))
 
    print(
        f"Epoch {epoch:02d} | Loss: {total_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e} | "
        f"Train Acc: {train_acc:.4f} F1: {train_f1:.4f} Prec: {train_prec:.4f} Rec: {train_rec:.4f} | "
        f"Val Acc: {val_acc:.4f} F1: {val_f1:.4f} Prec: {val_prec:.4f} Rec: {val_rec:.4f}"
    )

Epoch 01 | Loss: 0.4908 | LR: 5.00e-04 | Train Acc: 0.6610 F1: 0.0177 Prec: 0.0090 Rec: 0.6431 | Val Acc: 0.6609 F1: 0.0180 Prec: 0.0091 Rec: 0.6481
Epoch 02 | Loss: 0.3716 | LR: 4.98e-04 | Train Acc: 0.5917 F1: 0.0173 Prec: 0.0087 Rec: 0.7551 | Val Acc: 0.5917 F1: 0.0175 Prec: 0.0088 Rec: 0.7577
Epoch 03 | Loss: 0.3548 | LR: 4.96e-04 | Train Acc: 0.6284 F1: 0.0194 Prec: 0.0098 Rec: 0.7712 | Val Acc: 0.6282 F1: 0.0195 Prec: 0.0099 Rec: 0.7718
Epoch 04 | Loss: 0.3401 | LR: 4.92e-04 | Train Acc: 0.7618 F1: 0.0279 Prec: 0.0142 Rec: 0.7193 | Val Acc: 0.7617 F1: 0.0278 Prec: 0.0142 Rec: 0.7118
Epoch 05 | Loss: 0.3297 | LR: 4.88e-04 | Train Acc: 0.8798 F1: 0.0414 Prec: 0.0215 Rec: 0.5465 | Val Acc: 0.8797 F1: 0.0412 Prec: 0.0214 Rec: 0.5391
Epoch 06 | Loss: 0.3195 | LR: 4.83e-04 | Train Acc: 0.8679 F1: 0.0593 Prec: 0.0307 Rec: 0.8759 | Val Acc: 0.8677 F1: 0.0596 Prec: 0.0308 Rec: 0.8743
Epoch 07 | Loss: 0.3092 | LR: 4.77e-04 | Train Acc: 0.9056 F1: 0.0737 Prec: 0.0386 Rec: 0.7895 | Val Acc: 

## Table for Results

In [20]:
history = {
    "epoch": [],
    "train_acc": [],
    "train_f1": [],
    "train_precision": [],
    "train_recall": [],
    "val_acc": [],
    "val_f1": [],
    "val_precision": [],
    "val_recall": []
}


## Final Test Performance

In [21]:
test_acc, test_f1, test_prec, test_rec = evaluate(data.test_mask)
print(f"\n{'='*60}")
print(f"Final Test Results")
print(f"  Accuracy  : {test_acc:.4f}")
print(f"  F1        : {test_f1:.4f}")
print(f"  Precision : {test_prec:.4f}")
print(f"  Recall    : {test_rec:.4f}")
print(f"{'='*60}")
 
# ── Save results table 


Final Test Results
  Accuracy  : 0.9404
  F1        : 0.1309
  Precision : 0.0702
  Recall    : 0.9557


In [22]:
results_df = pd.DataFrame(history)
print("\nTraining history:")
print(results_df.to_string(index=False))


Training history:
Empty DataFrame
Columns: [epoch, train_acc, train_f1, train_precision, train_recall, val_acc, val_f1, val_precision, val_recall]
Index: []
